# Initialize

In [1]:
# @title Global Variables

from google.colab import drive
drive.mount('/content/drive')

!pip install scikit-learn -q
!pip install sqlitedict -q
!pip install openai -q

import os
from sqlitedict import SqliteDict
import hashlib
import json

working_dir = "/content/drive/MyDrive/LLM Auction/"
dataset_dir = "/content/drive/MyDrive/LLM Auction/dataset/"
logprobs_cache_dir = working_dir + "cache_logprobs/"
lmexaminer_cache_dir = working_dir + "cache_lmexaminer/"

current_directory = os.getcwd()
print(f"current_directory: {current_directory}")

os.makedirs(logprobs_cache_dir, exist_ok=True)
os.makedirs(lmexaminer_cache_dir, exist_ok=True)

# Cache key function for caching the output
def generate_cache_key(params):
    '''
    # generate a unique cache key based on the request parameters
    ## md5 seems to be enough
    params: dict, request parameters
    '''
    params_string = json.dumps(params, sort_keys=True)
    return hashlib.sha256(params_string.encode('utf-8')).hexdigest()

# @markdown - Select a debug mode (0 for no log, 1 for light log, 2 for detailed log)
debug_mode = 0 # @param [0, 1, 2]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
current_directory: /content


# Utils

In [2]:
def process_corpus(text):
    lines = text.strip().split('\n')
    clean_lines = []
    ad_slot_positions = []

    for index, line in enumerate(lines):
        if line.strip().lower() == '[ad slot]':
            ad_slot_positions.append(len(clean_lines))
        else:
            clean_lines.append(line)

    return clean_lines, ad_slot_positions


def insert_ad(lines, ad_slot_position, ad):
    merged = lines.copy()

    merged.insert(ad_slot_position, ad)

    return merged


def insert_ads(lines, ad_slot_positions, ads):
    if len(ad_slot_positions) != len(ads):
        raise ValueError("ad_slot_positions and ads must be the same length.")

    merged = lines.copy()

    # Zip positions with ads, sort by position descending
    insertions = sorted(zip(ad_slot_positions, ads), reverse=True)

    for pos, ad in insertions:
        merged.insert(pos, ad)

    return merged

def extract_json(text):
    match = re.search(r'(\{.*\})', text, re.DOTALL)
    if match:
        json_str = match.group(1)
        return json_str
    else:
        print("Can not find JSON")
        return None

# LMExaminer

In [3]:
# @title Define the LMExaminerCalculator Class

import openai
import threading
import re

class LMExaminerCalculator:
    def __init__(self, model_name="deepseek/deepseek-r1", cache_dir="./cache_lmexaminer/", debug_mode=1):
        self.model_name = model_name
        self.debug_mode = debug_mode

        # Initialize the API client (using OpenRouter as in your code)
        try:
            self.openrouter_client = openai.OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key="" # Replace with your actual key if different
            )
            self.openrouter_chat = self.openrouter_client.chat.completions
        except Exception as e:
            print(f"Error initializing OpenAI client for LMExaminer: {e}")
            self.openrouter_client = None
            self.openrouter_chat = None


        # Initialize cache
        os.makedirs(cache_dir, exist_ok=True)
        cache_filename = f"lmexaminer_{(model_name).replace('/', '--')}.sqlite"
        self.lmexaminer_db = SqliteDict(os.path.join(cache_dir, cache_filename), autocommit=True)
        self.cache_lock = threading.Lock() # Lock for safe cache operations

        self.system_prompt = """
You are an expert in digital advertising and user experience. Your task is to rate how suitable different ad genres would be if inserted into a specific location within an LLM-generated response.
"""

        self.user_prompt_template = """
Context: {}
User Query: {}

The LLM response contains an ad slot (marked as [Ad Slot]) where an advertisement could be inserted.

Here is the text surrounding Ad Slot:

{}
[Ad Slot] (THIS IS WHERE THE AD WOULD BE INSERTED)
{}

For each of the following ad genres, rate the suitability of inserting an ad from that genre into this slot on a scale from 1 (poor) to 5 (excellent).

When rating, consider:
- Fluency: Would the ad fit naturally within the local flow of the text?
- Coherence: Would the ad align logically with the context of the response?

Here are the ad genres to rate:

Ad Genres:
1. Airlines - Examples: Flight deals, airline promotions
2. Apparel - Examples: Clothing, shoes, accessories
3. Automotive - Examples: Cars, motorcycles, EV vehicles
4. Electronics - Examples: Smartphones, computers, tablets
5. FMCG (Fast-Moving Consumer Goods) - Examples: Personal care products, household items
6. Finance - Examples: Banking services, insurance, credit cards
7. Hotels - Examples: Hotel chains, booking services
8. Media - Examples: Streaming services, social media platforms
9. Packaged Food - Examples: Snacks, beverages, prepared meals
10. Restaurants - Examples: Fast food, cafes, dining establishments

For each genre, provide:
1. An explanation for your rating according to the above instruction.
2. A rating from 1-5.

1 – Poor: Strongly irrelevant; disrupts or confuses the reader.
2 – Weak: Loosely related but still feels misplaced or intrusive.
3 – Fair: Marginal fit; tolerable but clearly not natural.
4 – Good: Generally coherent and contextually appropriate, but not perfectly natural (e.g., there are other better positions to allocate this ad genre).
5 – Excellent: Only use for seamless, highly relevant, and contextually natural — feels native around the above and below sentences.

Format your response as JSON with this structure:
{{
  "ratings": {{
    "Airlines": {{ "explanation": "...", "score": X }},
    "Apparel": {{ "explanation": "...", "score": X }},
    ...
  }}
}}

Ensure your ratings are justified based on the context and the natural flow of the text.
"""

    def generate_cache_key(self, params):
        '''
        # generate a unique cache key based on the request parameters
        ## md5 seems to be enough
        params: dict, request parameters
        '''
        params_string = json.dumps(params, sort_keys=True)
        return hashlib.sha256(params_string.encode('utf-8')).hexdigest()

    def call_api(self, params, use_cache=True):
        if self.openrouter_chat is None:
            print("API client not initialized. Cannot make API call.")
            return None

        key = self.generate_cache_key(params)
        with self.cache_lock:
            if use_cache and (key in self.lmexaminer_db):
                if self.debug_mode > 1: print(f"Cache hit for LMExaminer key: {key}")
                return self.lmexaminer_db[key]
        if self.debug_mode > 1: print(f"Cache miss for LMExaminer key: {key}")


        try:
            response = self.openrouter_chat.create(**params)
            if response.choices is None:
                print(f"Error calling API: {response.error['message']}")
                return None
            content = response

            with self.cache_lock:
                if use_cache:
                    self.lmexaminer_db[key] = content
            return content
        except Exception as e:
            print(f"An error occurred during API call: {e}")
            return None

    def simple_call_api(self, system_text, user_text, max_tokens=4000, temperature=0):
        messages = [{
            "role": "system",
            "content": system_text,
        }, {
            "role": "user",
            "content": user_text,
        }] if system_text != "" else [{
            "role": "user",
            "content": user_text,
        }]
        params = {
            "model": self.model_name,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "frequency_penalty": 0,
            "presence_penalty": 0
        }
        return self.call_api(params)

    def extract_json(self, text):
        match = re.search(r'(\{.*\})', text, re.DOTALL)
        if match:
            json_str = match.group(1)
            return json.loads(json_str)
        else:
            if self.debug_mode > 0: print("Warning: Could not find JSON in LMExaminer response.")
            return None


    def get_suitability_ratings(self, context, user_query, response_with_slots, slot_index):
        """
        Gets suitability ratings for all ad genres from the LM-examiner for a specific ad slot.

        Args:
            context (str): The overall context of the interaction.
            user_query (str): The user's original query.
            response_with_slots (str): The LLM response containing [Ad Slot] markers.
            slot_index (int): The 0-based index of the ad slot to examine.

        Returns:
            dict: A dictionary where keys are ad genres and values are dictionaries
                  containing 'explanation' and 'score', or None if API call or JSON extraction fails.
        """
        if self.openrouter_client is None:
             print("LMExaminer API client not initialized. Cannot get ratings.")
             return None

        # Use the process_corpus from your utils or adapt it here
        lines, ad_slot_positions = process_corpus(response_with_slots)

        if slot_index < 0 or slot_index >= len(ad_slot_positions):
            print(f"Error: slot_index {slot_index} is out of bounds for {len(ad_slot_positions)} ad slots.")
            return None

        target_pos_in_clean_lines = ad_slot_positions[slot_index]

        text_before_ad_slot = "\n".join(lines[:target_pos_in_clean_lines])
        text_after_ad_slot = "\n".join(lines[target_pos_in_clean_lines:])


        user_prompt_formatted = self.user_prompt_template.format(
            context,
            user_query,
            text_before_ad_slot,
            text_after_ad_slot
        )

        api_response_text = self.simple_call_api(
            self.system_prompt,
            user_prompt_formatted,
        )

        if api_response_text:
            json_data = self.extract_json(api_response_text.choices[0].message.content)
            if json_data and "ratings" in json_data:
                return json_data["ratings"]
            else:
                if self.debug_mode > 0: print("Could not extract valid JSON ratings from API response.")
                return None
        else:
            if self.debug_mode > 0: print("API call returned no response text.")
            return None


    def calculate_coherence(self, context, user_query, response_with_slots, ads_to_insert):
        """
        Calculates a combined coherence score based on LM-examiner ratings for the inserted ads.
        This method assumes the score is the sum of the suitability ratings for the
        specific ad genres inserted at their respective locations.

        Args:
            context (str): The overall context of the interaction.
            user_query (str): The user's original query.
            response_with_slots (str): The LLM response containing [Ad Slot] markers.
            ads_to_insert (list of tuples): A list of (ad_genre, slot_index) tuples,
                                           where slot_index is 0-based.

        Returns:
            float: The combined LM-examiner suitability score for the inserted ads,
                   or None if ratings cannot be retrieved for any of the insertions.
        """
        total_suitability_score = 0.0
        processed_slots = set()

        for ad_genre, slot_index in ads_to_insert:
            # Only call the examiner once per slot if multiple ads target the same slot
            if slot_index not in processed_slots:
                suitability_ratings = self.get_suitability_ratings(
                    context,
                    user_query,
                    response_with_slots,
                    slot_index
                )
                processed_slots.add(slot_index)

                if suitability_ratings is None:
                    if self.debug_mode > 0:
                        print(f"Warning: Could not retrieve ratings for slot index {slot_index}. Skipping this insertion.")
                    # Depending on desired behavior, you might return None immediately or handle this differently
                    continue # Skip this specific insertion, continue with others if possible

            # Find the score for the specific ad_genre at this slot
            if suitability_ratings and ad_genre in suitability_ratings:
                score = suitability_ratings[ad_genre].get("score", 0.0) # Use .get with default 0.0 in case score is missing
                total_suitability_score += score
                if self.debug_mode > 0:
                    print(f"Added score {score} for '{ad_genre}' at slot {slot_index}")
            else:
                 if self.debug_mode > 0:
                     print(f"Warning: Could not find rating for ad genre '{ad_genre}' in slot {slot_index} ratings.")


        if self.debug_mode > 0:
            print(f"Total LM-Examiner Suitability Score: {total_suitability_score}")

        return total_suitability_score

# Example

parameters: total slot number N, max slot to use K, bid/coherence ratio alpha

assume we have an oracle coherence measurement, takes in the organic response with slots, and a list ads_to_insert, including (ad genre, slot number)

input: user prompt, organic response with slots, list of bidder info (bid, preferred ad genres list, ad content)

output: response with ad inserted

run a brute force to find the maximized total bid + alpha * coherence measurement

### Input Variables


In [4]:
ad_genre = '''Airlines
Apparel
Automotive
Electronics
FMCG
Finance
Hotels
Media
Packaged Food
Restaurants'''

ad_genre_list = ad_genre.split("\n")
ad_genre_to_id = {genre: idx for idx, genre in enumerate(ad_genre_list)}

ad_genre_to_id["FMCG (Fast-Moving Consumer Goods)"] = 4
ad_genre_to_id["Fast-Moving Consumer Goods"] = 4

user_prompt = '''Give me brief suggestions for a trip to New York City'''

organic_response = '''
Here are some brief suggestions for a trip to **New York City**:

[Ad Slot]

### Must-See Attractions
[Ad Slot]
* **Statue of Liberty & Ellis Island** – Iconic landmarks; consider booking a morning ferry.
* **Central Park** – Great for walking, biking, or a picnic; visit Bethesda Terrace or Bow Bridge.
* **Empire State Building / Top of the Rock** – For panoramic city views.
* **Times Square** – Best experienced at night for the lights and energy.
* **9/11 Memorial & Museum** – A moving, well-curated site.

### Culture & Entertainment
[Ad Slot]
* **Broadway Show** – Get tickets in advance or try same-day discounts via TKTS.
* **Metropolitan Museum of Art (The Met)** and **MoMA** – Both world-class.
* **Chelsea Galleries** – For contemporary art lovers.
* **Jazz Clubs in Harlem or Greenwich Village** – For authentic NYC nightlife.

### Food Highlights
[Ad Slot]
* **Pizza:** Joe’s Pizza, Prince Street Pizza.
* **Bagels:** Ess-a-Bagel, Russ & Daughters.
* **Fine dining:** Katz’s Deli for classic pastrami, Le Bernardin for Michelin-level seafood.
* **Markets:** Chelsea Market or Smorgasburg (weekends).

### Neighborhood Walks
[Ad Slot]
* **SoHo** – Shopping & architecture.
* **Greenwich Village** – Bohemian charm & music history.
* **Williamsburg (Brooklyn)** – Trendy cafés, thrift stores, skyline views.
* **Chinatown & Little Italy** – Great food and culture.

### Tips
[Ad Slot]
* Get a **MetroCard** or use **OMNY** (tap-to-pay) for subway rides.
* Walk as much as possible—NYC is very pedestrian-friendly.
* Book museum and show tickets ahead of time.

'''


### Utils

In [5]:
# @title Compute Coherence
import numpy as np

def processing_coherence_matrix(user_prompt, response_with_slots):

    model_name = "deepseek/deepseek-r1" #"openai/gpt-5"
    lmexaminer_calculator = LMExaminerCalculator(model_name=model_name, cache_dir=lmexaminer_cache_dir)

    lines, ad_slot_pos = process_corpus(response_with_slots)

    n_ad_slots = len(ad_slot_pos)
    n_ad_genres = len(ad_genre_list)
    results = np.zeros((n_ad_slots, n_ad_genres))

    for slot_index, pos in enumerate(ad_slot_pos):

        # Get suitability ratings for all ad genres at this slot
        suitability_ratings = lmexaminer_calculator.get_suitability_ratings(
            "None",
            user_prompt,
            response_with_slots,
            slot_index
        )

        if suitability_ratings:
            # Store the score for each ad genre at this position
            # Need to map the ad genre string from LM Examiner to the ad_genre_id in your results key
            # Assuming ad_genre_list contains the exact strings the LM Examiner uses

            for ad_genre, details in suitability_ratings.items():
                if ad_genre in ad_genre_to_id:
                    ad_genre_id = ad_genre_to_id[ad_genre]
                    results[slot_index, ad_genre_id] = float(details["score"])
                else:
                    print(f"Warning: LM-Examiner returned rating for unknown genre '{ad_genre}' in Slot {slot_index}")

        else:
            print(f"Could not get LM-examiner ratings for Slot {slot_index}")

    if np.sum(results == 0) > 0:
        print(f"Warning: LM-Examiner returned no ratings for {np.sum(results == 0)} slots.")

    return results


In [6]:
# @title Compute Matching Matrix

def compute_matching_matrix(bids, coherence_matrix):
    """
    Computes the matching matrix based on bids and coherence matrix.

    Args:
        bids: A numpy array of bids over each genre, shape = (n_bidders, n_genres).
        coherence_matrix: A numpy array representing the coherence matrix, shape = (n_slots, n_genres).
        alpha: The weight for the coherence matrix.

    Returns:
        A numpy array representing the matching matrix, shape = (n_bidders, n_slots).
    """
    matching_matrix = np.dot(np.array(bids, dtype=np.float64), np.array(coherence_matrix, dtype=np.float64).T)

    return matching_matrix


In [7]:
# @title VCG Mechanism with JV

from scipy.optimize import linear_sum_assignment

def vcg_assignment(V, K):
    """
    VCG (Clarke pivot) payments for one-to-one assignment.

    Args:
      V: (n_bidders x n_items) valuations; no -inf entries (all pairs allowed).
         Unmatched = outside option 0.
      K: exactly assign K items (must satisfy 0 <= K <= min(n_bidders, n_items)).

    Returns:
      matches: list of length n_bidders with item index or None (if unmatched)
      payments: list of length n_bidders with VCG payment for each bidder
      welfare: total welfare of allocation (sum of assigned bidders' valuations)
    """
    V = np.asarray(V, dtype=float)
    n_bidders, n_items = V.shape

    def solve_exact_k(V, K):
        """
        Solve welfare-maximizing assignment selecting exactly K real items,
        """
        n_bidders, n_items = V.shape

        if n_items > K:
            V_pad = np.concat([V, np.ones((n_items - K, n_items)) * 1000000000], axis = 0)
        else:
            V_pad = V

        row_ind, col_ind = linear_sum_assignment(-V_pad)

        # Extract matches for REAL bidders to REAL items only
        matches = [None] * n_bidders
        welfare = 0.0
        for r, c in zip(row_ind, col_ind):
            if r < n_bidders and c < n_items:
                matches[r] = c
                welfare += V[r, c]
        return matches, welfare

    # --- 1) Welfare-maximizing allocation with all bidders (exactly K items)
    matches, welfare = solve_exact_k(V, K)

    # --- 2) Compute VCG payments (Clarke pivot)
    payments = [0.0] * n_bidders
    # Value each bidder gets in the chosen allocation (0 if unmatched)
    bidder_value = [0.0] * n_bidders
    for i, j in enumerate(matches):
        if j is not None:
            bidder_value[i] = V[i, j]

    # Total welfare with everyone:
    W = welfare

    for i in range(n_bidders):
        # Remove bidder i and recompute optimal welfare for others
        V_minus_i = V.copy()
        V_minus_i[i,:] -= 1000000000
        _, W_minus_i = solve_exact_k(V_minus_i, K)

        # Clarke pivot: payment = (welfare of others without i) - (welfare of others with i)
        payments[i] = W_minus_i - (W - bidder_value[i])

    return matches, payments, welfare

# vcg_assignment(V=compute_matching_matrix(bids, preferences, coherence_matrix), K=10)

### Compute Assignment and Payment

In [18]:
def compute_assignment_and_payment(K, user_prompt, organic_response, bidder_info):

    coherence_matrix = (np.array(processing_coherence_matrix(user_prompt, organic_response)) - 1.0) / 4.0

    bids = np.array([[bidder["bids"][genre] if genre in bidder["bids"] else 0 for genre in ad_genre_list] for bidder in bidder_info])

    n_bidders, n_genres = bids.shape
    n_slots, _ = coherence_matrix.shape

    matching_matrix = compute_matching_matrix(bids, coherence_matrix)
    matches, payments, welfare = vcg_assignment(V=matching_matrix, K=K)

    print(coherence_matrix)
    # print(matching_genres)
    # print(matches, payments, welfare)
    matched_bids = [
        bids[i] if matched is not None else None
        for i, matched in enumerate(matches)
    ]

    matched_coherence = [
        coherence_matrix[matched] if matched is not None else None
        for i, matched in enumerate(matches)
    ]

    return matches, matched_bids, matched_coherence, payments, welfare


for X in range(0, 15):
    bidder_info = [
        {'bids': {'Hotels':X*0.5, 'Airlines':X*0.5}, 'content': '[Sponsored: Book your trip with Alpha Trip]'},
        {'bids': {'Electronics':20}, 'content': '[Sponsored: Get the latest tech at Beta Buy]'},
        {'bids': {'Apparel':5, 'Hotels':5, 'Restaurants':10}, 'content': '[Sponsored: Discover local gems with Gamma Map]'},
    ]
    matches, matched_bids, matched_coherence, payments, welfare = \
    compute_assignment_and_payment(2, user_prompt, organic_response, bidder_info)
    print(f"================= Bid of Alpha Trip: {X} =================")

    lines, ad_slot_pos = process_corpus(organic_response)

    ads_positions = []
    ads_to_insert = []

    for i, bidder in enumerate(bidder_info):
        if matches[i] is None:
            continue

        ads_to_insert.append(bidder['content'] + f" with bid {(str(matched_bids[i]))}, coherence = {str(matched_coherence[i])}, welfare = {str(np.sum(matched_bids[i] * matched_coherence[i]))})")
        ads_positions.append(ad_slot_pos[matches[i]])

    print("\n".join(insert_ads(lines, ads_positions, ads_to_insert)))
    print("\n")

    for i, payment in enumerate(payments):
        if matches[i] is None:
            print(f"Bidder {i}'s payment: 0")
        else:
            print(f"Bidder {i}'s payment: {payment}")
    print("\n\n\n")



[[1.   0.5  0.   0.25 0.   0.5  1.   0.25 0.5  0.75]
 [0.75 0.25 0.   0.25 0.   0.25 1.   0.   0.5  0.75]
 [0.25 0.   0.   0.   0.   0.   0.5  0.25 0.25 0.75]
 [0.25 0.   0.   0.   0.25 0.   0.5  0.   0.75 1.  ]
 [0.25 0.75 0.   0.   0.   0.   0.5  0.   0.25 1.  ]
 [0.75 0.25 0.   0.25 0.   0.5  1.   0.   0.25 0.25]]
================= Bid of Alpha Trip: 0 =================
Here are some brief suggestions for a trip to **New York City**:


### Must-See Attractions
* **Statue of Liberty & Ellis Island** – Iconic landmarks; consider booking a morning ferry.
* **Central Park** – Great for walking, biking, or a picnic; visit Bethesda Terrace or Bow Bridge.
* **Empire State Building / Top of the Rock** – For panoramic city views.
* **Times Square** – Best experienced at night for the lights and energy.
* **9/11 Memorial & Museum** – A moving, well-curated site.

### Culture & Entertainment
* **Broadway Show** – Get tickets in advance or try same-day discounts via TKTS.
* **Metropolitan Museu